
# Local 3-WL on the CFI graphs $G_3$ and $H_3$

This notebook turns the proof sketch into an executable example.

We do three things:

1. construct the CFI graphs $G_3$ and $H_3$,
2. implement the **local 3-WL** refinement algorithm,
3. run local 3-WL on both graphs and check that it distinguishes them.


In [1]:

import itertools
from collections import Counter, defaultdict
from math import comb

import networkx as nx



## 1. CFI construction

For each base edge $e$ we create two edge-vertices $e^0,e^1$.

For each base vertex $v$ we create vertices $(v,S)$ where $S$ is:

- an **even** subset of the incident edges of $v$ in $G_k$,
- an **odd** subset for the distinguished vertex in $H_k$,
- an **even** subset for all other base vertices in $H_k$.

Then we connect $(v,S)$ to:

- $e^1$ if $e\in S$,
- $e^0$ if $e\notin S$.

This is exactly the construction described in the proof sketch.


In [2]:

def get_incident_edges(K, v):
    return sorted(tuple(sorted(e)) for e in K.edges() if v in e)


def get_subsets_by_parity(edges, parity):
    subsets = []
    n = len(edges)

    for mask in range(2 ** n):
        subset = frozenset(edges[j] for j in range(n) if mask & (1 << j))
        if len(subset) % 2 == parity:
            subsets.append(subset)

    return subsets


def build_cfi_graph(K, distinguished_vertex=None):
    G = nx.Graph()

    # Step 1: for each base edge e, add e^0 and e^1 and connect them
    for e in K.edges():
        e_sorted = tuple(sorted(e))
        e0 = (e_sorted, 0)
        e1 = (e_sorted, 1)

        G.add_node(e0, node_type="edge", base_edge=e_sorted, bit=0)
        G.add_node(e1, node_type="edge", base_edge=e_sorted, bit=1)
        G.add_edge(e0, e1)

    # Step 2: for each base vertex v, add the subset vertices (v, S)
    for v in K.nodes():
        incident_edges = get_incident_edges(K, v)

        # H_k flips parity only at the distinguished vertex
        parity = 1 if v == distinguished_vertex else 0

        for S in get_subsets_by_parity(incident_edges, parity):
            subset_node = (v, S)
            G.add_node(subset_node, node_type="subset", base_vertex=v, subset=S)

            # Connect (v, S) to e^1 if e in S, else to e^0
            for e in incident_edges:
                bit = 1 if e in S else 0
                G.add_edge(subset_node, (e, bit))

    return G


In [3]:

k = 3
K = nx.complete_graph(k + 1)

G3 = build_cfi_graph(K, distinguished_vertex=None)
H3 = build_cfi_graph(K, distinguished_vertex=0)

print(f"Base graph: K_{k+1}")
print(f"  number of base vertices = {K.number_of_nodes()}")
print(f"  number of base edges    = {K.number_of_edges()}")

print("\nConstructed graphs:")
print(f"  G_3: {G3.number_of_nodes()} vertices, {G3.number_of_edges()} edges")
print(f"  H_3: {H3.number_of_nodes()} vertices, {H3.number_of_edges()} edges")
print(f"  same size: {G3.number_of_nodes() == H3.number_of_nodes() and G3.number_of_edges() == H3.number_of_edges()}")


Base graph: K_4
  number of base vertices = 4
  number of base edges    = 6

Constructed graphs:
  G_3: 28 vertices, 54 edges
  H_3: 28 vertices, 54 edges
  same size: True


### Vertex count check

For the base graph $K_{k+1}$:

- each base vertex has degree $k$,
- the number of even subsets of its incident edges is $2^{k-1}$,
- there are $k+1$ base vertices,
- and each base edge contributes two edge-vertices.

So the total number of vertices should be

$$
(k+1)\cdot 2^{k-1} + 2\binom{k+1}{2}.
$$

For $k=3$, this becomes

$$
4\cdot 2^2 + 2\binom{4}{2} = 16 + 12 = 28.
$$

In [4]:

expected_vertices = (k + 1) * (2 ** (k - 1)) + 2 * comb(k + 1, 2)

print("Expected vertex count:", expected_vertices)
print("Actual |V(G_3)|      :", G3.number_of_nodes())
print("Actual |V(H_3)|      :", H3.number_of_nodes())
print("Count check passed   :", G3.number_of_nodes() == expected_vertices == H3.number_of_nodes())


Expected vertex count: 28
Actual |V(G_3)|      : 28
Actual |V(H_3)|      : 28
Count check passed   : True



## 2. Distance-two structure behind the proof

The proof sketch focuses on the special subset vertices of the form $(v,\emptyset)$.

- In $G_3$, all four vertices $(0,\emptyset),(1,\emptyset),(2,\emptyset),(3,\emptyset)$ exist.
- In $H_3$, the vertex $(0,\emptyset)$ does **not** exist, because vertex $0$ uses **odd** subsets.

First we check this directly.


In [5]:

print("Vertices of the form (v, ∅):\n")
for v in K.nodes():
    node = (v, frozenset())
    print(
        f"  ({v}, ∅): in G_3 = {node in G3.nodes()}, in H_3 = {node in H3.nodes()}"
    )


Vertices of the form (v, ∅):

  (0, ∅): in G_3 = True, in H_3 = False
  (1, ∅): in G_3 = True, in H_3 = True
  (2, ∅): in G_3 = True, in H_3 = True
  (3, ∅): in G_3 = True, in H_3 = True


In [6]:

empty_vertices_G = [(v, frozenset()) for v in K.nodes()]

print("Pairwise distances among the empty-subset vertices in G_3:\n")
all_two = True

for i, u in enumerate(empty_vertices_G):
    for j, v in enumerate(empty_vertices_G):
        if i < j:
            dist = nx.shortest_path_length(G3, u, v)
            ok = "✓" if dist == 2 else "✗"
            all_two = all_two and (dist == 2)
            print(f"  d(({u[0]}, ∅), ({v[0]}, ∅)) = {dist}  {ok}")

print("\nAll pairwise distances are 2:", all_two)


Pairwise distances among the empty-subset vertices in G_3:

  d((0, ∅), (1, ∅)) = 2  ✓
  d((0, ∅), (2, ∅)) = 2  ✓
  d((0, ∅), (3, ∅)) = 2  ✓
  d((1, ∅), (2, ∅)) = 2  ✓
  d((1, ∅), (3, ∅)) = 2  ✓
  d((2, ∅), (3, ∅)) = 2  ✓

All pairwise distances are 2: True



The proof sketch is really about choosing **one subset vertex per base vertex** and asking whether all of them can be pairwise at distance $2$.

That is slightly more specific than just asking for an arbitrary distance-two clique in the whole graph.
The helper below searches for the largest such family under the condition:

- at most one chosen subset vertex for each base vertex.


In [7]:

def subset_vertices(X):
    return [v for v, data in X.nodes(data=True) if data["node_type"] == "subset"]


def max_one_per_base_distance_two_family(X):
    """
    Search for the largest family of subset vertices such that:
    1. any two chosen vertices are at graph distance exactly 2,
    2. we choose at most one subset vertex for each base vertex.
    """
    subsets = subset_vertices(X)

    by_base = defaultdict(list)
    for node in subsets:
        base_vertex = X.nodes[node]["base_vertex"]
        by_base[base_vertex].append(node)

    base_vertices = sorted(by_base.keys())
    dist2 = dict(nx.all_pairs_shortest_path_length(X, cutoff=2))
    best = []

    def backtrack(index, chosen):
        nonlocal best

        # Simple pruning
        if len(chosen) + (len(base_vertices) - index) <= len(best):
            return

        if index == len(base_vertices):
            if len(chosen) > len(best):
                best = chosen.copy()
            return

        current_base = base_vertices[index]

        # Option 1: skip this base vertex
        backtrack(index + 1, chosen)

        # Option 2: choose one subset vertex for this base vertex
        for candidate in by_base[current_base]:
            ok = True
            for other in chosen:
                if dist2.get(candidate, {}).get(other) != 2:
                    ok = False
                    break

            if ok:
                chosen.append(candidate)
                backtrack(index + 1, chosen)
                chosen.pop()

    backtrack(0, [])
    return best


In [8]:

family_G = max_one_per_base_distance_two_family(G3)
family_H = max_one_per_base_distance_two_family(H3)

print("Largest one-per-base distance-two family in G_3:")
for node in family_G:
    print(" ", node)
print("size =", len(family_G))

print("\nLargest one-per-base distance-two family in H_3:")
for node in family_H:
    print(" ", node)
print("size =", len(family_H))


Largest one-per-base distance-two family in G_3:
  (0, frozenset())
  (1, frozenset())
  (2, frozenset())
  (3, frozenset())
size = 4

Largest one-per-base distance-two family in H_3:
  (1, frozenset())
  (2, frozenset())
  (3, frozenset())
size = 3



For the $k=3$ example, this matches the intended intuition of the proof sketch:

- in $G_3$ we can pick one subset vertex for each of the four base vertices,
- in $H_3$ the best such family has size only $3$.

Now we move to the main algorithmic part: local $3$-WL.


## 3. Local $k$-WL

For a $k$-tuple

$$
\mathbf{v}=(v_1,\dots,v_k),
$$

the **local** version only replaces coordinate $i$ by a **neighbor** of $v_i$.

So for each coordinate $i$, we look at

$$
\phi_i(\mathbf{v},w)=(v_1,\dots,v_{i-1},w,v_{i+1},\dots,v_k)
\quad\text{for } w\in N(v_i).
$$

### Initialization

At round $0$, each tuple gets its **atomic type**:

- which coordinates are equal,
- which pairs are adjacent.

### Refinement

At later rounds, the new color depends on:

- the old color of the tuple,
- the multiset of colors reachable by local moves in each coordinate.

To compare colors across **both graphs**, we always compress the signatures **jointly** on $G_3$ and $H_3$.
That way, a color ID means the same thing in both graphs.

In [9]:

def atomic_type(graph, tup):
    k = len(tup)

    equality_pattern = tuple(
        tuple(tup[i] == tup[j] for j in range(k))
        for i in range(k)
    )

    adjacency_pattern = tuple(
        tuple((i != j) and graph.has_edge(tup[i], tup[j]) for j in range(k))
        for i in range(k)
    )

    return (equality_pattern, adjacency_pattern)


def compress_to_ids(values):
    codebook = {}
    ids = []

    for value in values:
        if value not in codebook:
            codebook[value] = len(codebook)
        ids.append(codebook[value])

    return ids, codebook


def local_k_wl_two_graphs(G, H, k, max_iter=20):
    tuples_G = list(itertools.product(G.nodes(), repeat=k))
    tuples_H = list(itertools.product(H.nodes(), repeat=k))

    # ----- round 0: atomic types -----
    raw_initial = [atomic_type(G, t) for t in tuples_G] + [atomic_type(H, t) for t in tuples_H]
    initial_ids, _ = compress_to_ids(raw_initial)

    colors_G = dict(zip(tuples_G, initial_ids[:len(tuples_G)]))
    colors_H = dict(zip(tuples_H, initial_ids[len(tuples_G):]))

    history = [(colors_G.copy(), colors_H.copy())]

    # ----- refinement rounds -----
    for iteration in range(1, max_iter + 1):
        signatures = []

        for graph, tuples_list, colors in [(G, tuples_G, colors_G), (H, tuples_H, colors_H)]:
            for t in tuples_list:
                local_signature = []

                for i in range(k):
                    neighbor_colors = [
                        colors[t[:i] + (w,) + t[i+1:]]
                        for w in graph.neighbors(t[i])
                    ]
                    local_signature.append(tuple(sorted(neighbor_colors)))

                signatures.append((colors[t], tuple(local_signature)))

        signature_ids, _ = compress_to_ids(signatures)

        new_colors_G = dict(zip(tuples_G, signature_ids[:len(tuples_G)]))
        new_colors_H = dict(zip(tuples_H, signature_ids[len(tuples_G):]))

        history.append((new_colors_G.copy(), new_colors_H.copy()))

        if new_colors_G == colors_G and new_colors_H == colors_H:
            return {
                "colors_G": new_colors_G,
                "colors_H": new_colors_H,
                "history": history,
                "iterations": iteration - 1,
                "tuple_count_G": len(tuples_G),
                "tuple_count_H": len(tuples_H),
            }

        colors_G, colors_H = new_colors_G, new_colors_H

    return {
        "colors_G": colors_G,
        "colors_H": colors_H,
        "history": history,
        "iterations": max_iter,
        "tuple_count_G": len(tuples_G),
        "tuple_count_H": len(tuples_H),
    }


In [10]:
result = local_k_wl_two_graphs(G3, H3, k=3, max_iter=20)

colors_G = result["colors_G"]
colors_H = result["colors_H"]

stable_counter_G = Counter(colors_G.values())
stable_counter_H = Counter(colors_H.values())

print("Local 3-WL finished.")
print(f"  tuples in G_3: {result['tuple_count_G']}")
print(f"  tuples in H_3: {result['tuple_count_H']}")
print(f"  refinement rounds until stabilization: {result['iterations']}")
print(f"  number of stable colors in G_3: {len(stable_counter_G)}")
print(f"  number of stable colors in H_3: {len(stable_counter_H)}")

Local 3-WL finished.
  tuples in G_3: 21952
  tuples in H_3: 21952
  refinement rounds until stabilization: 8
  number of stable colors in G_3: 185
  number of stable colors in H_3: 185


If the two stable color multisets are different, then local $3$-WL distinguishes the graphs.

To connect this with the proof sketch, we also inspect the tuple

$$
P=((1,\emptyset),(2,\emptyset),(3,\emptyset)) \in V(G_3)^3.
$$

If the proof idea is reflected by the computation, then the stable color of $P$ should not appear on any tuple of $H_3$.

In [11]:

P = ((1, frozenset()), (2, frozenset()), (3, frozenset()))
color_P = colors_G[P]

present_in_H = color_P in set(colors_H.values())
count_in_G = sum(1 for c in colors_G.values() if c == color_P)
count_in_H = sum(1 for c in colors_H.values() if c == color_P)

print("Witness tuple P =", P)
print("Stable color of P in G_3:", color_P)
print("Does this color appear anywhere in H_3?", present_in_H)
print("How many tuples in G_3 have this color?", count_in_G)
print("How many tuples in H_3 have this color?", count_in_H)


Witness tuple P = ((1, frozenset()), (2, frozenset()), (3, frozenset()))
Stable color of P in G_3: 164
Does this color appear anywhere in H_3? False
How many tuples in G_3 have this color? 192
How many tuples in H_3 have this color? 0


### Final histogram comparison

The tuple $P$ is a useful witness suggested by the proof sketch, but **local $3$-WL distinguishes the graphs by comparing the full stable color histograms on all $3$-tuples**.

So after the refinement stabilizes, we compare the multiplicity of every stable color in $G_3^3$ and $H_3^3$.
If these histograms are different, then local $3$-WL distinguishes the graphs.

In [12]:
all_colors = sorted(set(stable_counter_G) | set(stable_counter_H))
different_colors = []

print("Stable color histogram comparison")
print("-" * 36)
print(f"total color IDs appearing in at least one graph: {len(all_colors)}")

for color in all_colors:
    count_G = stable_counter_G.get(color, 0)
    count_H = stable_counter_H.get(color, 0)
    if count_G != count_H:
        different_colors.append((color, count_G, count_H))

print(f"number of colors with different multiplicities: {len(different_colors)}")
print(f"histograms equal? {stable_counter_G == stable_counter_H}")

if different_colors:
    print("\nColors whose multiplicities differ:")
    print(f"{'color':>8} {'count in G_3':>14} {'count in H_3':>14}")
    for color, count_G, count_H in different_colors:
        print(f"{color:>8} {count_G:>14} {count_H:>14}")

Stable color histogram comparison
------------------------------------
total color IDs appearing in at least one graph: 370
number of colors with different multiplicities: 370
histograms equal? False

Colors whose multiplicities differ:
   color   count in G_3   count in H_3
       0             12              0
       1             12              0
       2             96              0
       3             24              0
       4             48              0
       5             48              0
       6             96              0
       7             12              0
       8             12              0
       9             96              0
      10             24              0
      11             48              0
      12             48              0
      13             96              0
      14             96              0
      15             96              0
      16             96              0
      17             96              0
      18             9

## 4. Conclusion

This notebook implements the full pipeline for the **$k=3$** case:

- it constructs the CFI graphs $G_3$ and $H_3$ from $K_4$,
- it checks the main structural features used in the proof sketch,
- it implements **local 3-WL**,
- it inspects the witness tuple $P=((1,\emptyset),(2,\emptyset),(3,\emptyset))$,
- and it finally compares the full stable color histograms on all $3$-tuples.

The key algorithmic conclusion is the last one: the stable color histograms are **not equal**, so **local 3-WL distinguishes $G_3$ and $H_3$**.